In [1]:
import numpy as np
from datetime import datetime

In [2]:
# Loading CSV data:
data = np.genfromtxt(
    "../data/data.csv",
    delimiter=",",
    dtype=str,
    skip_header=1
)

# Checking Preview:
print("Raw Data:\n", data[:5])

Raw Data:
 [['U01' 'reading' '2024-01-01 07:05' 'done']
 ['U01' 'reading' '2024-01-02 07:10' 'done']
 ['U01' 'reading' '2024-01-03 07:25' 'done']
 ['U01' 'reading' '2024-01-04 07:40' 'missed']
 ['U01' 'reading' '2024-01-05 07:15' 'done']]


In [3]:
# str -> Data: 
def convertTime(tsArray):
    dtList = []
    for t in tsArray:
        dt = datetime.strptime(t, "%Y-%m-%d %H:%M")
        dtList.append(dt)
    dtArray = np.array(dtList)

    numericTimeList = []
    for t in dtArray:
        diff = t - dtArray[0]
        seconds = diff.total_seconds()
        hours = seconds / 3600
        numericTimeList.append(hours)
    numericTime = np.array(numericTimeList)

    return numericTime


In [4]:
# TimeStamp:
timestamps = data[:, 2]

print("Timestamps:\t", timestamps[:3])

Timestamps:	 ['2024-01-01 07:05' '2024-01-02 07:10' '2024-01-03 07:25']


In [5]:
timeNumeric = convertTime(timestamps)
timeNumeric = np.round(timeNumeric, 2)

print("Hours since first event:\t", timeNumeric[:5])


Hours since first event:	 [ 0.   24.08 48.33 72.58 96.17]


In [6]:
status = data[:, 3]
print("Sorted Status:\n", status[:10])


Sorted Status:
 ['done' 'done' 'done' 'missed' 'done' 'done' 'done' 'missed' 'done' 'done']


In [7]:
#  Inter-Event Time Gaps 
gaps = np.diff(timeNumeric, prepend=timeNumeric[0])

print("Hours Since Previous Event:\n", gaps[:10])

Hours Since Previous Event:
 [ 0.   24.08 24.25 24.25 23.59 23.83 24.42 24.33 23.33 24.17]


In [8]:
# Habit Timeline
nEvents = len(status)

currentStreak = 0
longestStreak = 0
failureCount  = 0
fatigueScore  = 0

stateLog = np.zeros((nEvents, 5))
print("Total Events:", nEvents)


Total Events: 30


In [15]:
#  Strict Streak Rule 
def strictRule(action, streak):
    if action == "done":
        return streak + 1, False 
    else:
        return 0, True 


In [ ]:
# Run Simulation:
for i in range(nEvents):
    action = status[i]

    # Applying Streak Rule:
    newStreak, broke = strictRule(action, currentStreak)

    # Fatigue Logic:
    if action == "missed":
        fatigueScore += 1
        failureCount += 1
    else:
        fatigueScore = max(0, fatigueScore - 0.5)

    # Updating Streak: 
    currentStreak = newStreak
    longestStreak = max(longestStreak, currentStreak)

    if broke:
        recoveryFlag = 1 
    else:
        recoveryFlag = 0

    # Log State:
    stateLog[i] = [timeNumeric[i], currentStreak, failureCount, fatigueScore, recoveryFlag]
    
print("State Log (first 10 rows):\n")
print(stateLog[:10])

State Log (first 10 rows):

[[  0.     1.     7.     0.5    0.  ]
 [ 24.08   2.     7.     0.     0.  ]
 [ 48.33   3.     7.     0.     0.  ]
 [ 72.58   0.     8.     1.     1.  ]
 [ 96.17   1.     8.     0.5    0.  ]
 [120.     2.     8.     0.     0.  ]
 [144.42   3.     8.     0.     0.  ]
 [168.75   0.     9.     1.     1.  ]
 [192.08   1.     9.     0.5    0.  ]
 [216.25   2.     9.     0.     0.  ]]


In [17]:
streaks  = stateLog[:, 1]
fatigue  = stateLog[:, 3]
collapse = stateLog[:, 4]

print("\n-----HabitFlow Results-----\n")
print("___________________________")
print("| Longest Streak:  |", longestStreak,"   |")
print("| Total Failures:  |", int(failureCount),"  |")
print("| Collapse Events: |", int(np.sum(collapse)),"   |")
print("| Average Fatigue: |", round(np.mean(fatigue), 2),"|")
print("___________________________")


-----HabitFlow Results-----

___________________________
| Longest Streak:  | 4    |
| Total Failures:  | 14   |
| Collapse Events: | 7    |
| Average Fatigue: | 0.35 |
___________________________


In [18]:
totalEvents = len(status) 
totalDone = np.sum(status == "done")
consistencyScore = ( (totalDone / totalEvents) * 70 + (longestStreak / totalEvents) * 30)

print("Consistency Score:", round(consistencyScore, 2))

Consistency Score: 57.67


In [21]:
def assignGrade(consistencyScore):
    if consistencyScore >= 85:
        return "A"
    elif consistencyScore >= 70:
        return "B"
    elif consistencyScore >= 50:
        return "C"
    elif consistencyScore >= 30:
        return "D"
    else:
        return "E"
    
grade = assignGrade(consistencyScore)

print("Performance Grade:", grade)


Performance Grade: C


In [23]:
def getMotivation(grade):

    quotes = {
        "A": "Elite consistency. You’re operating like a machine. Protect this discipline.",
        "B": "Strong habits forming. A little more consistency can make you unstoppable.",
        "C": "You’re trying, but inconsistency is breaking momentum. Focus on streak protection.",
        "D": "Your habit system is unstable. Reduce friction and restart small.",
        "E": "Reset required. Start with micro-habits and rebuild discipline gradually."
    }
    return quotes.get(grade, "Keep pushing forward.")

motivation = getMotivation(grade)

print("Motivational Insight:", motivation)


Motivational Insight: You’re trying, but inconsistency is breaking momentum. Focus on streak protection.


In [24]:
print("Habit Flow Final:\n")

print(f"Consistency Score : ", round(consistencyScore,2))
print(f"Performance Grade : ", grade)
print(f"Longest Streak    : ", longestStreak)
print("\nMotivation: ", motivation)


Habit Flow Final:

Consistency Score :  57.67
Performance Grade :  C
Longest Streak    :  4

Motivation:  You’re trying, but inconsistency is breaking momentum. Focus on streak protection.
